## 한국어 뉴스 주제 분류 파인튜닝

기본 모델: `klue/bert-base`  
데이터셋: `klue/ynat`  
결과: 한국어 뉴스 제목을 주제별로 분류

파인튜닝 전에는 기본 모델이 뉴스 주제 분류용으로 학습되어 있지 않기 때문에 원하는 라벨을 제대로 예측하지 못한다.  
파인튜닝 후에는 한국어 뉴스 제목을 입력했을 때 정치, 경제, 사회, 생활문화, 세계, IT과학, 스포츠 중 하나로 분류할 수 있다.

영화리뷰 감정분석
기본 모델: beomi/kcbert-base
데이터셋: NSMC
분류 라벨: 부정, 긍정
결과: 한국어 영화 리뷰의 감정을 분류

In [31]:
!pip install -q -U "datasets<4.0.0" transformers evaluate accelerate

In [32]:
import datasets
print(f"datasets library version: {datasets.__version__}")

datasets library version: 3.6.0


In [33]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate

print("GPU 사용 가능 여부:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

GPU 사용 가능 여부: True
GPU 이름: Tesla T4


데이터 불러오기

In [34]:
from datasets import load_dataset

# put code here

# 데이터 불러오기
dataset = load_dataset("yangwang825/klue-ynat")

print(dataset)
print(dataset["train"][0])

# 라벨 정보 확인
label_names = dataset["train"].features["label"].names
num_labels = len(label_names)

print("라벨 개수:", num_labels)
print("라벨 목록:", label_names)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 9107
    })
})
{'text': '유튜브 내달 2일까지 크리에이터 지원 공간 운영', 'label': 3}
라벨 개수: 7
라벨 목록: ['IT과학', '경제', '사회', '생활문화', '세계', '스포츠', '정치']


In [ ]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print(train_dataset)
print(test_dataset)

기본 모델과 토크나이저 불러오기

In [35]:
# put code here

# 사전학습 모델 선택
# 한국어 뉴스 분류이므로 KLUE 계열 모델 사용
model_checkpoint = "klue/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 라벨 매핑
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [36]:
# 토큰화 함수
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)


print(tokenized_dataset)

Map:   0%|          | 0/9107 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9107
    })
})


In [38]:
# 평가 지표
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy.compute(
        predictions=predictions,
        references=labels
    )

    f1_score = f1.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )

    return {
        "accuracy": acc["accuracy"],
        "f1": f1_score["f1"]
    }

In [42]:
# 학습 설정
training_args = TrainingArguments(
    output_dir="./ynat_news_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    report_to="none"
)

In [50]:
# Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics
)

# 학습 시작
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.375726,0.432936,0.852531,0.859434
2,0.296283,0.386648,0.871527,0.869707


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.375726,0.432936,0.852531,0.859434
2,0.296283,0.386648,0.871527,0.869707
3,0.200243,0.439199,0.867355,0.866918


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8565, training_loss=0.31555037412181347, metrics={'train_runtime': 1680.6896, 'train_samples_per_second': 81.534, 'train_steps_per_second': 5.096, 'total_flos': 4507097372985600.0, 'train_loss': 0.31555037412181347, 'epoch': 3.0})

In [51]:
# 최종 평가
eval_result = trainer.evaluate()
print(eval_result)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.200243,0.386648,3,0.871527,0.869707


{'eval_loss': 0.3866477906703949, 'eval_accuracy': 0.8715273965081806, 'eval_f1': 0.8697066342952535}


In [52]:
# 모델 저장
save_path = "./ynat_news_classifier_final"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("모델 저장 완료:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

모델 저장 완료: ./ynat_news_classifier_final


In [53]:
# 파이프라인으로 테스트
news_classifier = pipeline(
    task="text-classification",
    model=save_path,
    tokenizer=save_path,
    device=0 if torch.cuda.is_available() else -1
)

test_titles = [
    "삼성전자, 차세대 반도체 기술 공개",
    "손흥민 시즌 첫 해트트릭 폭발",
    "정부, 부동산 공급 대책 발표",
    "신작 영화 개봉 첫날 박스오피스 1위",
    "한국은행 기준금리 동결 결정"
]

for title in test_titles:
    result = news_classifier(title)
    print(title)
    print(result)
    print()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

삼성전자, 차세대 반도체 기술 공개
[{'label': 'IT과학', 'score': 0.8680112957954407}]

손흥민 시즌 첫 해트트릭 폭발
[{'label': '스포츠', 'score': 0.9970009922981262}]

정부, 부동산 공급 대책 발표
[{'label': '정치', 'score': 0.4739774167537689}]

신작 영화 개봉 첫날 박스오피스 1위
[{'label': '생활문화', 'score': 0.9836733341217041}]

한국은행 기준금리 동결 결정
[{'label': '경제', 'score': 0.987718939781189}]

